In [ ]:
# -*- coding: utf-8 -*-

# Анализ фильмов: что влияет на сборы и рейтинг
**Дисциплина:** Анализ данных на Python — итоговый проект

**Команда:** _впишите участников_

**Источник данных:** The Movie Database (TMDB) API — https://www.themoviedb.org/
*This product uses the TMDB API but is not endorsed or certified by TMDB.*

**Исследовательские вопросы:**
1. Связан ли бюджет фильма с его кассовыми сборами?
2. Влияет ли хронометраж на рейтинг?
3. Различается ли рейтинг между жанрами?

**Как устроен ноутбук:** сбор данных через API (с пагинацией и дозапросом деталей) → очистка и создание новых признаков → разведочный анализ → проверка гипотез статтестами → визуализация → выводы.

## 0. Подготовка окружения
Все библиотеки уже предустановлены в Google Colab — отдельная установка не нужна.

In [ ]:
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from getpass import getpass
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110

### Ключ TMDB API

In [ ]:
API_KEY = getpass("Вставьте ваш TMDB API key (v3): ").strip()
BASE_URL = "https://api.themoviedb.org/3"
assert API_KEY, "Ключ не введён!"
print("Ключ принят.")

import requests
test = requests.get("https://api.themoviedb.org/3/movie/550",
                    params={"api_key": API_KEY}, timeout=15)
print("HTTP статус:", test.status_code)
print(test.json())

## 1. Сбор данных
Подход в два шага (это и есть «сложный сбор»):
1. **Пагинация** по эндпоинту `/discover/movie` — собираем ID самых популярных фильмов с нескольких десятков страниц.
2. **Дозапрос деталей** по каждому ID через `/movie/{id}` — получаем бюджет, сборы, хронометраж и жанры (этих полей нет в выдаче discover).

In [ ]:
session = requests.Session()


def tmdb_get(endpoint, params=None, retries=3):
    """Запрос к TMDB API с базовой обработкой ошибок и рейт-лимита."""
    params = dict(params or {})
    params["api_key"] = API_KEY
    url = f"{BASE_URL}{endpoint}"
    for _ in range(retries):
        try:
            resp = session.get(url, params=params, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code == 429:        # превышен лимит запросов
                time.sleep(2)
                continue
            return None
        except requests.RequestException:
            time.sleep(1)
    return None

def collect_movie_ids(n_pages=30, min_votes=100):
    """Шаг 1: пагинация по /discover/movie, собираем уникальные ID фильмов."""
    movie_ids = []
    for page in range(1, n_pages + 1):
        data = tmdb_get("/discover/movie", {
            "sort_by": "vote_count.desc",   # самые "массовые" фильмы
            "vote_count.gte": min_votes,
            "page": page,
            "language": "en-US",
        })
        if data and data.get("results"):
            movie_ids.extend(m["id"] for m in data["results"])
        time.sleep(0.05)
    return list(dict.fromkeys(movie_ids))    # убираем дубликаты, сохраняя порядок


movie_ids = collect_movie_ids(n_pages=30)   # ~600 фильмов
print(f"Собрано уникальных ID: {len(movie_ids)}")

def fetch_movie_details(movie_id):
    """Шаг 2: детали по одному фильму."""
    d = tmdb_get(f"/movie/{movie_id}")
    if not d:
        return None
    return {
        "id": d.get("id"),
        "title": d.get("title"),
        "release_date": d.get("release_date"),
        "budget": d.get("budget"),
        "revenue": d.get("revenue"),
        "runtime": d.get("runtime"),
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "popularity": d.get("popularity"),
        "original_language": d.get("original_language"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])),
    }


records = []
for mid in tqdm(movie_ids, desc="Загрузка деталей фильмов"):
    rec = fetch_movie_details(mid)
    if rec:
        records.append(rec)
    time.sleep(0.05)

df_raw = pd.DataFrame(records)
print("Размер сырого датасета:", df_raw.shape)
df_raw.head()

df_raw.to_csv("movies_raw.csv", index=False)
print("Сырые данные сохранены в movies_raw.csv")

## 2. Очистка и предобработка
Что делаем: приводим типы, помечаем нули в `budget`/`revenue`/`runtime` как пропуски (в TMDB 0 = «нет данных»), убираем дубликаты, создаём новые расчётные признаки и заполняем пропуски хронометража медианой (статистический метод).

In [ ]:
df = df_raw.copy()

# 1. Типы данных: дата релиза -> datetime, отдельный столбец года
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year

# 2. Нули в финансовых полях = отсутствующие данные -> NaN
for col in ["budget", "revenue", "runtime"]:
    df.loc[df[col] == 0, col] = np.nan

# 3. Дубликаты
df = df.drop_duplicates(subset="id").reset_index(drop=True)

# 4. Работа с жанрами
df["main_genre"] = df["genres"].apply(
    lambda s: s.split("|")[0] if isinstance(s, str) and s else np.nan)
df["n_genres"] = df["genres"].apply(
    lambda s: len(s.split("|")) if isinstance(s, str) and s else 0)

# 5. Новые расчётные признаки
df["profit"] = df["revenue"] - df["budget"]
df["roi"] = df["revenue"] / df["budget"]                 # окупаемость
df["decade"] = (df["year"] // 10 * 10).astype("Int64")
df["is_english"] = (df["original_language"] == "en").astype(int)
df["is_profitable"] = (df["profit"] > 0).astype("Int64")

# 6. Пропуски хронометража -> медиана (статистический метод заполнения)
df["runtime"] = df["runtime"].fillna(df["runtime"].median())

print("Пропуски по столбцам после очистки:")
print(df.isna().sum())
df.describe(include="number")

## 3. Разведочный анализ (EDA)
Смотрим на данные в разрезах: средний рейтинг и сборы по жанрам и по десятилетиям.

### 3.1. Статистика по жанрам
Сгруппируем фильмы по основному жанру и посчитаем для каждого количество фильмов, средний рейтинг и средние сборы. Фильтр `n >= 10` отсекает жанры с 1–2 фильмами — по ним статистика ненадёжна. Сортировка по убыванию рейтинга поможет сразу увидеть лидеров и аутсайдеров.

In [ ]:
genre_stats = (df.groupby("main_genre")
                 .agg(n=("id", "size"),
                      avg_rating=("vote_average", "mean"),
                      avg_revenue=("revenue", "mean"))
                 .query("n >= 10")
                 .sort_values("avg_rating", ascending=False))
genre_stats

**Вывод:** Лидеры по среднему рейтингу — **Drama (7.79)** и **Crime (7.68)**, аутсайдеры — **Science Fiction (7.12)** и **Action (7.13)**. При этом по средним сборам больше всех зарабатывают **Family (880 млн долларов)** и **Adventure (705 млн долларов)**, а меньше всех — **Romance (191 млн долларов)** и **Horror (204 млн долларов)**. Это значит, что коммерческий успех и зрительская оценка — разные вещи: драмы любят критики, а семейные и приключенческие фильмы — касса.

### 3.2. Статистика по десятилетиям
Посмотрим, как менялись средний рейтинг и средний хронометраж фильмов от десятилетия к десятилетию. Это поможет увидеть тренды: становятся ли фильмы длиннее, и как меняется зрительская оценка со временем.

In [ ]:
decade_stats = (df.dropna(subset=["decade"])
                  .groupby("decade")
                  .agg(n=("id", "size"),
                       avg_rating=("vote_average", "mean"),
                       avg_runtime=("runtime", "mean")))
decade_stats

**Вывод:** Хронометраж стабильно растёт — с 83 мин в 1930-х до 134 мин в 2020-х, фильмы становятся длиннее. Средний рейтинг снижается от ~8.1 (1970-е) до 7.16 (2010-е), но это частично эффект выжившего: из старых фильмов в топ-600 по числу оценок попали только лучшие. По 1950–60-м в выборке единичные фильмы, поэтому по ним выводы делать рано.

### 3.3. Корреляционная матрица
Построим тепловую карту попарных корреляций Пирсона между всеми числовыми признаками. Это покажет, какие переменные связаны сильнее всего, и поможет выбрать подходящие статистические тесты для проверки гипотез.

In [ ]:
plt.figure(figsize=(10, 8))
num_cols = ["budget", "revenue", "runtime", "vote_average",
            "vote_count", "popularity", "n_genres", "profit"]
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0,
            fmt=".2f", square=True)
plt.title("Корреляционная матрица числовых переменных")
plt.tight_layout()
plt.show()

**Что видно:** сильнее всего связаны бюджет и сборы (0.73). Любопытно, что
бюджет **отрицательно** связан с рейтингом (−0.37): дорогие фильмы оценивают
ниже. Рейтинг заметнее всего тянут популярность (0.41) и число голосов (0.40). Блокбастеры обсуждают активнее.

### 3.4. Матрица диаграмм рассеивания (pairplot)
Визуализируем попарные зависимости между бюджетом, сборами, хронометражем и рейтингом. На диагонали — распределения отдельных переменных (KDE), вне диагонали — scatter-графики. Это позволяет увидеть нелинейные связи и выбросы, которые не видны в матрице корреляций Пирсона.

In [ ]:
g = sns.pairplot(df[["budget", "revenue", "runtime", "vote_average"]].dropna(),
                 diag_kind="kde", plot_kws=dict(alpha=0.35, s=14, color="#0F2C68"))
g.fig.suptitle("Матрица диаграмм рассеивания", y=1.02)
plt.show()

**Вывод:** На scatter-графиках видно:
- **Бюджет vs Сборы** — чёткий восходящий тренд, но с большим разбросом (есть фильмы с маленьким бюджетом и огромными сборами).
- **Хронометраж vs Рейтинг** — слабый восходящий тренд, разброс огромный.
- **Распределения бюджета и сборов** — сильно скошены вправо (большинство фильмов с малыми значениями, единичные блокбастеры с огромными). Это подтверждает, что распределения ненормальные.

## 4. Проверка распределений на нормальность

### Тест Шапиро-Уилка на нормальность
Перед выбором статистических тестов нужно проверить, нормально ли распределены наши переменные. От этого зависит, какой тест использовать:
- Если распределение нормальное - параметрический тест **Пирсона**.
- Если ненормальное - непараметрический тест **Спирмена**.

Применяем тест Шапиро-Уилка (H₀: распределение нормальное) и считаем асимметрию (skewness) и эксцесс (kurtosis) для каждого признака.

In [ ]:
num_cols = ["budget", "revenue", "runtime", "vote_average"]
print(f"{'Признак':<14}{'Шапиро W':>10}{'p-value':>12}{'Асимметрия':>13}{'Эксцесс':>10}  Вывод")
for col in num_cols:
    s = df[col].dropna()
    w, p = stats.shapiro(s)
    verdict = "нормально" if p > 0.05 else "НЕ нормально"
    print(f"{col:<14}{w:>10.3f}{p:>12.2e}{s.skew():>13.2f}{s.kurtosis():>10.2f}  {verdict}")

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color="#0F2C68")
    ax.set_title(col)
plt.tight_layout(); plt.show()

**Вывод:** Все четыре признака **не проходят** тест на нормальность (p < 0.05 для всех):

| Признак | Асимметрия | Эксцесс | Характер |
|---------|-----------|---------|----------|
| budget | 0.92 | 0.97 | Правосторонняя асимметрия, тяжёлые хвосты |
| revenue | 1.92 | 6.38 | Сильная правосторонняя асимметрия, очень тяжёлые хвосты |
| runtime | 0.91 | 1.64 | Правосторонняя асимметрия |
| vote_average | −0.31 | −0.35 | Слабая левосторонняя асимметрия, платикитическое |

**Что это значит для анализа:**
- Бюджет и сборы имеют «длинные правые хвосты» - большинство фильмов дешёвые, но есть единичные блокбастеры с бюджетами в сотни миллионов.
- Для связи «бюджет - сборы» формально нужен Спирмен, но Пирсон всё равно приводим как ориентир (связь линейная, r = 0.69).
- Для связи «хронометраж - рейтинг» однозначно используем Спирмен - распределения ненормальные, связь не обязана быть линейной.

## 5. Проверка гипотез
Применяем статистические тесты из курса. Уровень значимости α = 0.05.

**Гипотеза 1. Бюджет связан с кассовыми сборами.**
Обе переменные количественные, поэтому используем корреляцию Пирсона.
H₀: связи нет (r = 0). H₁: связь есть. Уровень значимости α = 0.05.

In [ ]:
sub = df.dropna(subset=["budget", "revenue"])
r, p = stats.pearsonr(sub["budget"], sub["revenue"])
print(f"H1. Бюджет vs сборы: r = {r:.3f}, p = {p:.2e}, n = {len(sub)}")
print("Вывод:", "связь значима" if p < 0.05 else "связь не значима")

**Результат:** r = 0.69, p < 0.001 (n = 594). Корреляция Пирсона не может работать с пропусками, ей нужны конкретные числа для обеих переменных. Поэтому 6 фильмов отбрасываются, так как не имеют данных о бюджете или о кассовых сборах.

Нулевую гипотезу отвергаем: связь значимая, положительная и умеренно-сильная — чем больше бюджет, тем выше сборы.

Однако r² ≈ 0.48 (коэффициент детерминации), то есть бюджет объясняет лишь ~48% дисперсии сборов - больше половины вариации объясняется другими факторами (жанр, маркетинг, дата релиза и т.д.).

**Гипотеза 2. Хронометраж связан с рейтингом.**
Используем корреляцию Спирмена: она работает с монотонной связью, устойчива к
выбросам и не требует линейности (рейтинг ограничен сверху).

In [ ]:
sub2 = df.dropna(subset=["runtime", "vote_average"])
rho, p2 = stats.spearmanr(sub2["runtime"], sub2["vote_average"])
print(f"H2. Хронометраж vs рейтинг: rho = {rho:.3f}, p = {p2:.2e}, n = {len(sub2)}")
print("Вывод:", "связь значима" if p2 < 0.05 else "связь не значима")

**Результат:** rho = 0.21, p < 0.001 (n = 600). Связь статистически значима, но
слабая: более длинные фильмы в среднем оцениваются чуть выше, однако
хронометраж - далеко не главный фактор рейтинга.

Статистическая значимость при таком малом rho объясняется большим размером выборки (n = 600) - даже слабые связи становятся значимыми при большом n.

**Гипотеза 3. Средний рейтинг различается между жанрами.**


**H₀:** средние рейтинги всех жанров равны. **H₁:** хотя бы один жанр отличается.

Используем однофакторный дисперсионный анализ (F-критерий, ANOVA) - он сравнивает средние количественного признака (рейтинг) в нескольких группах (жанры). Берём топ-5 самых частых жанров из `genre_stats`. Это ядро выборки (80% всех фильмов). Включать нишевые жанры с 2-5 фильмами было бы статистически некорректно, так как единичные выбросы исказили бы средние значения и дисперсию групп.

In [ ]:
top_genres = genre_stats.index[:5]
groups = [df.loc[df["main_genre"] == g, "vote_average"].dropna() for g in top_genres]
f_stat, p3 = stats.f_oneway(*groups)
print(f"H3. F-критерий по жанрам {list(top_genres)}:")
print(f"    F = {f_stat:.3f}, p = {p3:.2e}")
print("Вывод:", "различия значимы" if p3 < 0.05 else "различия не значимы")

**Результат:** F = 4.185, p = 0.003.

- p = 0.003 < 0.05 → нулевую гипотезу отвергаем, различия между жанрами значимы.

## 5.1. Линейная регрессия: бюджет - сборы

Корреляция показывает силу связи, но не её величину. Построим простую линейную регрессию, чтобы ответить на вопрос: **на сколько долларов увеличатся сборы при увеличении бюджета на 1$?**

Разделяем данные на обучающую (80%) и тестовую (20%) выборки, чтобы оценить качество модели на данных, которые она не видела при обучении.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

reg = df.dropna(subset=["budget", "revenue"])
X, y = reg[["budget"]], reg["revenue"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
pred = lr.predict(X_test)
print(f"R² (тест) = {r2_score(y_test, pred):.3f}")
print(f"RMSE = {mean_squared_error(y_test, pred)**0.5/1e6:.0f} млн $")
print(f"Коэффициент: каждый +1 $ бюджета → +{lr.coef_[0]:.2f} $ сборов")

**Результаты:**
- R² = 0.579 - модель объясняет 58% разброса сборов.
- RMSE = 261 млн \$ - средняя ошибка прогноза. Много, но учитывая, что средние сборы 463 млн \$, ошибка составляет 56% - приемлемо.
- Коэффициент = 3.55 - каждый вложенный доллар приносит в среднем 3.55$ сборов. То есть окупаемость ~3.5x.

Бюджет - полезный предиктор сборов, но R² = 0.58 означает, что 42% вариации объясняется другими факторами. Добавим их в множественную регрессию.

### 5.2. Множественная регрессия: бюджет + хронометраж + число голосов → сборы
Добавим к бюджету хронометраж и число голосов как контрольные переменные, чтобы проверить:
1. Увеличится ли R²?
2. Какие предикторы значимы (p < 0.05)?
3. Как изменится коэффициент при бюджете?

In [ ]:
import statsmodels.api as sm
reg2 = df.dropna(subset=["revenue", "budget", "runtime", "vote_count"])
Xm = sm.add_constant(reg2[["budget", "runtime", "vote_count"]])
ols = sm.OLS(reg2["revenue"], Xm).fit()
print(ols.summary())

**Множественная регрессия** (бюджет + хронометраж + число голосов) даёт
R² ≈ 0.56. По `.summary()` видно, какие предикторы значимы (p < 0.05) — главный
вклад у бюджета.

**Результаты:**
- **R² = 0.562** - модель объясняет ~56% дисперсии сборов (чуть меньше, чем простая регрессия с бюджетом - 0.579).
- **budget (coef = 3.30, p < 0.001)** - значим, каждый доллар бюджета приносит ~3.30$ сборов.
- **vote_count (coef = 20970, p < 0.001)** - значим, число голосов тоже предсказывает сборы (популярные фильмы зарабатывают больше).
- **runtime (coef = −385200, p = 0.458)** - **не значим**, хронометраж не вносит вклада при наличии бюджета и числа голосов.

**Главный вывод:** бюджет - главный предиктор сборов, число голосов - второй по важности, хронометраж - не значим.

## 6. Визуализация
Пять типов графиков, у каждого — заголовок, подписи осей и (где нужно) легенда.

In [ ]:
sub = df.dropna(subset=["budget", "revenue"])
plt.figure(figsize=(8, 6))
plt.scatter(sub["budget"] / 1e6, sub["revenue"] / 1e6, alpha=0.4, color="steelblue")
plt.xlabel("Бюджет, млн $")
plt.ylabel("Сборы, млн $")
plt.title("Связь бюджета и кассовых сборов")
plt.tight_layout()
plt.show()

va = df["vote_average"].dropna()
low = df.loc[df["vote_average"].idxmin()]
high = df.loc[df["vote_average"].idxmax()]

fig, ax = plt.subplots(figsize=(9, 5.5))
counts, edges, _ = ax.hist(va, bins=20, color="mediumpurple", edgecolor="white")
ymax = counts.max()

# линия среднего
ax.axvline(va.mean(), color="darkred", linestyle="--",
           label=f"Среднее = {va.mean():.2f}")

# подпись самого низкого рейтинга
ax.annotate(f"Минимум: {low['title']}\n({low['vote_average']:.1f})",
            xy=(low["vote_average"], 1),
            xytext=(low["vote_average"] + 0.15, ymax * 0.55),
            ha="left", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="#ffe0e0", ec="gray"),
            arrowprops=dict(arrowstyle="->", color="gray"))

# подпись самого высокого рейтинга
ax.annotate(f"Максимум: {high['title']}\n({high['vote_average']:.1f})",
            xy=(high["vote_average"], 1),
            xytext=(high["vote_average"] - 1.6, ymax * 0.75),
            ha="left", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="#e0ffe0", ec="gray"),
            arrowprops=dict(arrowstyle="->", color="gray"))

ax.set_xlabel("Средний рейтинг (vote_average)")
ax.set_ylabel("Количество фильмов")
ax.set_title("Распределение рейтингов популярных фильмов (топ-600 по числу оценок)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

order = genre_stats.sort_values("avg_rating", ascending=False)
plt.figure(figsize=(9, 5))
plt.bar(order.index, order["avg_rating"], color="teal")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Средний рейтинг")
plt.title("Средний рейтинг по основным жанрам (≥10 фильмов)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(decade_stats.index, decade_stats["avg_rating"], marker="o", color="darkorange",
         label="Средний рейтинг")
plt.xlabel("Десятилетие")
plt.ylabel("Средний рейтинг")
plt.title("Динамика среднего рейтинга по десятилетиям")
plt.legend()
plt.tight_layout()
plt.show()

top5 = list(genre_stats.index[:5])
data_box = [df.loc[df["main_genre"] == g, "vote_average"].dropna() for g in top5]
plt.figure(figsize=(9, 6))
plt.boxplot(data_box)
plt.xticks(range(1, len(top5) + 1), top5, rotation=45, ha="right")
plt.ylabel("Рейтинг")
plt.title("Разброс рейтингов по жанрам")
plt.tight_layout()
plt.show()

## 7. Выводы

Исследование проведено на выборке из 600 наиболее оцениваемых фильмов TMDB
(96.8% — англоязычные). Выборка смещена в сторону популярных картин, поэтому
97% фильмов в ней прибыльны — выводы стоит трактовать именно для «заметного»
кино, а не для индустрии в целом.

1. **Бюджет умеренно-сильно связан со сборами** (r = 0.69, p < 0.001). Дорогие
   фильмы в среднем зарабатывают больше, но большой разброс показывает, что
   деньги не гарантируют кассу.
2. **Хронометраж почти не влияет на рейтинг** (rho = 0.21, p < 0.001): связь
   значима, но слишком слабая, чтобы считать длину фильма фактором качества.
3. **Рейтинг значимо зависит от жанра** (F-критерий, p = 0.003): зрители выше ценят
   драмы и криминал, ниже — фантастику и боевики. При этом наибольшую кассу
   собирают семейные и анимационные фильмы — коммерческий успех и зрительская
   оценка не совпадают.
4. **Снижение среднего рейтинга к 2010-м** (с ~7.8–8.1 в 1970–80-х до 7.16)
   частично объясняется эффектом выжившего: из старых фильмов в топ попали лишь
   лучшие. По 1950–60-м в выборке единичные фильмы, поэтому выводы по ним
   осторожные.

**Практический смысл:** для прогноза кассовых сборов бюджет — полезный, но
недостаточный признак; для прогноза зрительской оценки важнее жанр, чем
бюджет или длительность.

**Ограничения:** только популярные фильмы; финансовые данные TMDB заполнены не
для всех картин; жанр упрощён до основного (первого в списке).